# Interfacing TopologicPy with Neo4j

In [1]:
# This cell is not needed if you have pip installed topologicpy
import sys
sys.path.append("C:/Users/sarwj/OneDrive - Cardiff University/Documents/GitHub/topologicpy/src")

## 1. Imports

In [2]:
from topologicpy.Neo4j import Neo4j
from topologicpy.GraphDB import GraphDB
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.CellComplex import CellComplex
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Graph import Graph
from topologicpy.Helper import Helper


## 2. Check the TopologicPy Version

In [3]:
print("This tutorial requires topologicpy version 0.9.31 or newer.")
print(Helper.Version())

This tutorial requires topologicpy version 0.9.31 or newer.
The version that you are using (0.9.31) is EQUAL TO the latest version available on PyPI.


## 3. Set your renderer
* Visual studio code: "vscode"
* Google Colab: "colab"
* Browser: "browser"

In [4]:
renderer = "vscode"

## 4. Create an Example Topologic Graph

In [5]:
cc = CellComplex.Prism(uSides=3, vSides=3, wSides=3)
cells = Topology.Cells(cc)

for i, c in enumerate(cells):
    if i == 5: # We are choosing one cell to make it special to identify it later.
        label = "Special"
        color = "purple"
    else:
        label = "Room_"+str(i).zfill(3)
        color = "red"
    d = Dictionary.ByKeysValues(
    ["label", "category", "name", "color", "id"],
    [label, "Space", label, color, label])
    c = Topology.SetDictionary(c, d)

# Create a graph of the cellComplex
g = Graph.ByTopology(cc)

# Set a graph id
graph_id = "test_graph"
d = Dictionary.ByKeyValue("id", graph_id)
g = Topology.SetDictionary(g, d)

# Get its vertices and edges
verts = Graph.Vertices(g)
edges = Graph.Edges(g)

# Specify what the edges mean to neo4j
for edge in edges:
    d = Dictionary.ByKeysValues(
    ["label", "category", "weight"],
    ["ADJACENT_TO", "Adjacency", 1.0])
    edge = Topology.SetDictionary(edge, d)

Topology.Show(g,
              vertexSize=20,
              vertexColorKey="color",
              backgroundColor="white",
              width=500,
              height=500,
              renderer=renderer)

In [6]:
from topologicpy.Graph import Graph
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from collections import Counter

vertices = Graph.Vertices(g)

ids = []
for i, v in enumerate(vertices):
    d = Topology.Dictionary(v)
    vid = Dictionary.ValueAtKey(d, "id", None)  # or "id", depending on your options
    label = Dictionary.ValueAtKey(d, "label", None)
    print(i, "vertex_id:", vid, "label:", label)
    ids.append(str(vid))

print("Duplicates:", [k for k, c in Counter(ids).items() if c > 1])

0 vertex_id: Room_000 label: Room_000
1 vertex_id: Room_001 label: Room_001
2 vertex_id: Room_002 label: Room_002
3 vertex_id: Room_003 label: Room_003
4 vertex_id: Room_004 label: Room_004
5 vertex_id: Special label: Special
6 vertex_id: Room_006 label: Room_006
7 vertex_id: Room_007 label: Room_007
8 vertex_id: Room_008 label: Room_008
9 vertex_id: Room_009 label: Room_009
10 vertex_id: Room_010 label: Room_010
11 vertex_id: Room_011 label: Room_011
12 vertex_id: Room_012 label: Room_012
13 vertex_id: Room_013 label: Room_013
14 vertex_id: Room_014 label: Room_014
15 vertex_id: Room_015 label: Room_015
16 vertex_id: Room_016 label: Room_016
17 vertex_id: Room_017 label: Room_017
18 vertex_id: Room_018 label: Room_018
19 vertex_id: Room_019 label: Room_019
20 vertex_id: Room_020 label: Room_020
21 vertex_id: Room_021 label: Room_021
22 vertex_id: Room_022 label: Room_022
23 vertex_id: Room_023 label: Room_023
24 vertex_id: Room_024 label: Room_024
25 vertex_id: Room_025 label: Room_02

## 5. Connect to your Neo4j instance. Make sure it is running first:
* If using the Desktop version, open Neo4j Desktop
* If using the online version (Aura), go to your console at: https://console.neo4j.io/

In [ ]:
from getpass import getpass

url = input("URL: ")
username = input("Username: ")
password = getpass("Password: ")

gdb_manager = Neo4j.Manager(url=url, username=username, password=password, database=None, silent=False)
print(gdb_manager)

## 6. Verify that the connection is working

* graphdb : dict
    The graph database descriptor.
* graph : topologic_core.Graph
    The input TopologicPy graph.
* graphIDKey : str , optional
    The graph dictionary key under which the graph id is stored. Default is None.
* vertexIDKey : str , optional
    The vertex dictionary key under which the vertex id is stored. Default is None.
* vertexLabelKey : str , optional
    The vertex dictionary key under which the vertex label is stored. Default is None.
* defaultVertexLabel : str , optional
    The default vertex label if no vertex label is found. Default is "Node".
* vertexCategoryKey : str , optional
    The vertex dictionary key used for category metadata. Default is "category".
* defaultVertexCategory : any , optional
    The default vertex category if none is found. Default is None.
* edgeLabelKey : str , optional
    The edge dictionary key under which the edge label/type is stored. Default is "label".
* defaultEdgeLabel : str , optional
    The default edge label if no edge label is found. Default is "CONNECTED_TO".
* edgeCategoryKey : str , optional
    The edge dictionary key used for category metadata. Default is "category".
* defaultEdgeCategory : any , optional
    The default edge category if none is found. Default is None.
* bidirectional : bool , optional
    If True, creates reverse edges where supported. Default is True.
* overwrite : bool , optional
    If True, overwrites an existing graph with the same id where supported. Default is True.

In [ ]:
graphdb = GraphDB.ByParameters(
    provider="neo4j",
    manager=gdb_manager,
    options={
        "database": None,
        "bidirectional": True,
        "overwrite": True,
        "graphIDKey": "id",
        "vertexIDKey": "id",
        "vertexLabelKey": "label",
        "defaultVertexLabel": "Node",
        "defaultVertexLabel": "Node",
        "vertexCategoryKey": "category",
        "defaultVertexCategory": None,
        "edgeLabelKey": "label",
        "defaultEdgeLabel": "CONNECTED_TO",
        "edgeCategoryKey": "category",
        "defaultEdgeCategory": None
    }
)

print("Graph DB:", graphdb)
records = Neo4j.Query(gdb_manager, "RETURN 1 AS n")
print("If everything is ok, the following line should print the number 1")
print(records[0]["n"])  # should print 1

## 7. Reset the Graph DB

In [ ]:
# Clear test DB
status = GraphDB.EmptyDatabase(graphdb)
print("Empty Database:", status)

# Ensure schema
ok = GraphDB.EnsureSchema(graphdb)
print("EnsureSchema:", ok)

## 8. Write the Topologic graph to Neo4j

In [ ]:
# This will write the topologic graph to Neo4j

graph_id = GraphDB.UpsertGraph(graphdb,
                               graph=g,
                               mantissa = 6,
                               silent = False
                               )
print(graph_id)

## 9. Inspect the result in Neo4j Dekstop or Aura DB Browser

Open Neo4j Browser and run the following line. Do NOT include the #:



In [ ]:
# MATCH (a)-[r]->(b) RETURN a, r, b

## 10. Test a few Cypher queries from Python

In [ ]:
records = GraphDB.Query(graphdb, "MATCH (n) RETURN count(n) AS c")
print("Node count:", records[0]["c"])

records = GraphDB.Query(graphdb, "MATCH ()-[r]->() RETURN count(r) AS c")
print("Relationship count:", records[0]["c"])

records = GraphDB.Query(
    graphdb,
    """
    MATCH (a)-[r]->(b)
    RETURN a.label AS from_name, type(r) AS rel_type, b.label AS to_name
    """
)

for record in records:
    print(record["from_name"], record["rel_type"], record["to_name"])

## 11. Return a Graph By Its ID

In [ ]:
g2 = GraphDB.GraphByID(graphdb, graph_id)
print(g2)
Topology.Show(g2,
              vertexSize=20,
              vertexColorKey="color",
              backgroundColor="white",
              width=500,
              height=500,
              renderer=renderer)

## 11. Inspect the Returned Graph

In [ ]:
vertices = Graph.Vertices(g2)
edges = Graph.Edges(g2)

print("Vertices:", len(vertices))
print("Edges:", len(edges))

for i, v in enumerate(vertices):
    d = Dictionary.PythonDictionary(Topology.Dictionary(v))
    print("Vertex", i, d)

for i, e in enumerate(edges):
    d = Dictionary.PythonDictionary(Topology.Dictionary(e))
    print("Edge", i, d)

## 12. Return Graphs by a Query

In [ ]:
# This query searches for node with degree = n_count and returns the sub-graph.
n_count = 3

graphs = GraphDB.GraphsByQuery(
    graphdb,
    query="""
    MATCH (n:Vertex)-[r:Edge]-(m:Vertex)
    WITH n, COUNT(DISTINCT m) AS degree
    WHERE degree = $degree
    MATCH (n)-[r:Edge]-(m:Vertex)
    RETURN n, r, m
    """,
    parameters={
        "degree": n_count,
    }
)

print("Queried graphs:", graphs)

for g3 in graphs:
    vertices = Graph.Vertices(g3)
    edges = Graph.Edges(g3)

    print("Vertices:", len(vertices))
    print("Edges:", len(edges))
    Topology.Show(g3,
                vertexSize=20,
                vertexColorKey="color",
                backgroundColor="white",
                width=500,
                height=500,
                renderer=renderer)


## 13. Find the "Special" node (purple) and return its neighborhood with k=2

In [ ]:
graphs = GraphDB.GraphsByQuery(
    graphdb,
    query="""
    MATCH (start:Vertex)
    WHERE start.label = "Special"
    MATCH p = (start)-[:Edge*1..2]-(nbr)
    UNWIND relationships(p) AS r
    UNWIND nodes(p) AS n
    RETURN DISTINCT n, r
    """,
    parameters={
        "degree": n_count,
    }
)

print("Queried graphs:", graphs)

for g4 in graphs:
    vertices = Graph.Vertices(g4)
    edges = Graph.Edges(g4)

    print("Vertices:", len(vertices))
    print("Edges:", len(edges))
    Topology.Show(g4,
                vertexSize=20,
                vertexColorKey="color",
                backgroundColor="white",
                width=500,
                height=500,
                renderer=renderer)


In [ ]:
import ast

s = '{"color": "red"}'


print(d)
print(type(d))

vertices = Graph.Vertices(g4)
edges = Graph.Edges(g4)

print("Vertices:", len(vertices))
print("Edges:", len(edges))

for i, v in enumerate(vertices):
    d = Topology.Dictionary(v)
    props = Dictionary.ValueAtKey(d, "props")
    props_d = ast.literal_eval(props)
    color = props_d['color']
    d = Dictionary.SetValueAtKey(d, "color", color)
    v = Topology.SetDictionary(v, d)

In [ ]:
Topology.Show(g4,
              vertexSize=20,
              vertexColorKey="color",
              backgroundColor="white",
              width=500,
              height=500,
              renderer=renderer)

## 13. Close Driver When Done

In [ ]:
Neo4j.Close(gdb_manager)

## 14. Using Neighborhood directly in TopologicPy for In-memory graphs

* Here we show you how to do it directly in TopologicPy
* k is the number of hops to explore from the selected vertex.


In [ ]:
g5 = Graph.Neigborhood(g, vertices=None, k=2, key="label", value="Special", searchType="equal")
Topology.Show(g5,
              vertexSize=20,
              vertexColorKey="color",
              backgroundColor="white",
              width=500,
              height=500,
              renderer=renderer)